# PhageMatch-PK — Step 6: Kleborate typing of the Pakistani *K. pneumoniae* cohort

**Runs on Google Colab.** The project laptop (8 GB RAM, no conda/WSL) can't install the
Kleborate/Kaptive stack, so genome typing happens here and only the results table comes back.

**Produces:** per genome — ST (MLST), **K-locus (capsule)**, **O-locus**, AMR determinants,
virulence scores. The K-locus is the one that matters most: it is the receptor that
*Klebsiella* phage depolymerases recognise, so the K-locus distribution *is* the phage target list.

---

### Run order

| Cell | What it does | Time |
|---|---|---|
| 1 | System tools + NCBI datasets CLI | ~1 min |
| 2 | pip install kleborate + kaptive | ~2 min |
| 3 | **Restart runtime** (mandatory) | ~30 s |
| 4 | Verify install, detect CLI syntax | ~1 min |
| 5 | Upload `kp_core_accessions.txt` | — |
| 6 | Download 260 genomes from NCBI | ~10 min |
| 7 | **Smoke test on one genome** | ~1 min |
| 8 | Full Kleborate run | 30–60 min |
| 9–11 | Collect, summarise, download results | ~1 min |

**Cell 3 restarts the runtime and that is expected** — `kaptive` needs numpy ≥ 2.4 and
numba ≥ 0.62, and Colab has already imported the older numpy into memory before you
start. Without the restart you get errors like *"numpy.dtype size changed"* or
*"module compiled against API version…"*. After the restart, **resume at Cell 4** — do
not re-run Cells 1–2.

## Cell 1 — System tools and the NCBI datasets CLI

In [ ]:
%%bash
# Deliberately NOT using `set -e`: if one probe fails we still want to see the
# rest of the diagnostics rather than losing the cell to the first error.

echo '--- system deps ---'
apt-get -qq update > /dev/null 2>&1
apt-get -qq install -y minimap2 ncbi-blast+ > /dev/null 2>&1

echo '--- NCBI datasets CLI ---'
curl -sSL -o /usr/local/bin/datasets \
  https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets
chmod +x /usr/local/bin/datasets

echo
echo '--- versions ---'
minimap2 --version || echo 'minimap2 MISSING'
datasets --version || echo 'datasets MISSING'
python -c 'import sys; print("python", sys.version.split()[0])'

## Cell 2 — Install Kleborate and Kaptive

Watch the numpy version printed before and after. If it changes, the restart in Cell 3
is not optional.

**Kaptive is pinned to 3.2.2 deliberately.** Kleborate 3.2.4 declares an unpinned
`kaptive` dependency, but Kaptive 3.3.0 restructured its package and removed
`kaptive.database`, which Kleborate's capsule module imports. Installing the latest
Kaptive produces `ModuleNotFoundError: No module named 'kaptive.database'` and Kleborate
will not start at all. 3.2.2 is the newest release that still ships that module.

In [ ]:
import subprocess, sys

# Do NOT unpin: Kaptive >=3.3.0 removed kaptive.database, which Kleborate imports.
KAPTIVE_PIN = "kaptive==3.2.2"

def numpy_version():
    r = subprocess.run([sys.executable, "-c", "import numpy; print(numpy.__version__)"],
                       capture_output=True, text=True)
    return r.stdout.strip() or "(not installed)"

before = numpy_version()
print(f"numpy before: {before}\n")

# Not quiet - if pip has to backtrack on numpy/numba we want to see it happen.
r = subprocess.run([sys.executable, "-m", "pip", "install", "kleborate", KAPTIVE_PIN],
                   capture_output=True, text=True)
print(r.stdout[-4000:])
if r.returncode != 0:
    print("\n!!! PIP FAILED !!!\n", r.stderr[-4000:])

after = numpy_version()
print(f"\nnumpy after: {after}")
print("\n>>> numpy CHANGED - you MUST run Cell 3 to restart." if before != after
      else "\n>>> numpy unchanged, but run Cell 3 anyway to be safe.")

## Cell 3 — Restart the runtime (mandatory)

This kills the kernel on purpose. Colab will show *"Your session crashed"* or
*"Runtime restarted"* — **that is the intended behaviour, not an error.**

**After it restarts, continue from Cell 4.** Do not re-run Cells 1–2 (installed packages
and the downloaded CLI survive the restart).

In [ ]:
import os

print("Restarting runtime now - this is expected. Continue at Cell 4.")
os.kill(os.getpid(), 9)

## Cell 4 — Verify the install and detect the CLI syntax

Kleborate v2 and v3 take different arguments, and v3's preset names have changed across
releases. Rather than assume, discover what this build accepts.

In [ ]:
import subprocess, sys

print("=== imports ===")
for mod in ["numpy", "numba", "kaptive", "kleborate"]:
    try:
        m = __import__(mod)
        print(f"  {mod:<10} {getattr(m, '__version__', 'ok')}")
    except Exception as e:
        print(f"  {mod:<10} FAILED: {type(e).__name__}: {e}")

print("\n=== kleborate --version ===")
v = subprocess.run(["kleborate", "--version"], capture_output=True, text=True)
print((v.stdout + v.stderr).strip())

print("\n=== kleborate --help ===")
h = subprocess.run(["kleborate", "--help"], capture_output=True, text=True)
help_text = h.stdout + h.stderr
print(help_text)

KLEBORATE_V3 = ("--preset" in help_text) or ("-p," in help_text) or ("--list-presets" in help_text)
print(">>> Kleborate v3 CLI" if KLEBORATE_V3 else ">>> Kleborate v2 CLI")

for flag in ["--list-presets", "--list-modules"]:
    if flag.strip("-").split("-")[-1] in help_text or flag in help_text:
        r = subprocess.run(["kleborate", flag], capture_output=True, text=True)
        out = (r.stdout + r.stderr).strip()
        if out:
            print(f"\n=== kleborate {flag} ===\n{out[:3000]}")

## Cell 5 — Upload the accession list

In [ ]:
from pathlib import Path
from google.colab import files

uploaded = files.upload()   # choose kp_core_accessions.txt
acc_file = Path(next(iter(uploaded)))

accessions = [ln.strip() for ln in acc_file.read_text().splitlines() if ln.strip()]
print(f"\n{len(accessions)} accessions loaded from {acc_file.name}")
print("first 5:", accessions[:5])

## Cell 6 — Download the assemblies from NCBI

Batched and resumable: completed batches are marked, so re-running after a dropped
connection only fetches what is missing.

In [ ]:
import shutil, subprocess, zipfile
from pathlib import Path

WORK = Path("/content/work")
ASM = WORK / "assemblies"
ASM.mkdir(parents=True, exist_ok=True)
BATCH = 25

batches = [accessions[i:i + BATCH] for i in range(0, len(accessions), BATCH)]
print(f"{len(batches)} batches of up to {BATCH}\n")

for n, batch in enumerate(batches, 1):
    marker = WORK / f".batch_{n}.done"
    if marker.exists():
        print(f"batch {n:>3}/{len(batches)}  cached")
        continue

    listfile = WORK / f"batch_{n}.txt"
    listfile.write_text("\n".join(batch) + "\n")
    zpath = WORK / f"batch_{n}.zip"

    res = subprocess.run(
        ["datasets", "download", "genome", "accession",
         "--inputfile", str(listfile), "--include", "genome",
         "--filename", str(zpath), "--no-progressbar"],
        capture_output=True, text=True,
    )
    if res.returncode != 0 or not zpath.exists():
        print(f"batch {n:>3} FAILED rc={res.returncode}: {res.stderr.strip()[:300]}")
        continue

    # Flatten to one .fna per accession, named by accession for clean joins later.
    with zipfile.ZipFile(zpath) as zf:
        for member in zf.namelist():
            if not member.endswith((".fna", ".fa", ".fasta")):
                continue
            parts = member.split("/")
            acc = parts[2] if len(parts) >= 3 else Path(member).stem
            with zf.open(member) as src, (ASM / f"{acc}.fna").open("wb") as dst:
                shutil.copyfileobj(src, dst)

    zpath.unlink()
    marker.touch()
    print(f"batch {n:>3}/{len(batches)}  ok  ({len(list(ASM.glob('*.fna')))} genomes so far)")

fastas = sorted(ASM.glob("*.fna"))
print(f"\nDownloaded {len(fastas)} / {len(accessions)} genomes "
      f"({sum(f.stat().st_size for f in fastas)/1e6:.0f} MB)")

missing = set(accessions) - {f.stem for f in fastas}
if missing:
    print(f"MISSING {len(missing)}: {sorted(missing)[:10]}")

## Cell 7 — Smoke test on a single genome

Do not skip this. It tries the candidate command forms on one genome and keeps whichever
works, so a CLI mismatch costs one minute instead of surfacing an hour into the full run.
If every variant fails, the full stderr is printed — send me that output.

In [ ]:
import subprocess
from pathlib import Path

TEST = WORK / "smoketest"
TEST.mkdir(exist_ok=True)
one = sorted(ASM.glob("*.fna"))[0]
print(f"test genome: {one.name}\n")

# Ordered by how likely each is to be the right syntax for a current build.
variants = [
    ("v3 preset kpsc",   ["kleborate", "-a", str(one), "-o", str(TEST / "t1"), "-p", "kpsc"]),
    ("v3 preset kp",     ["kleborate", "-a", str(one), "-o", str(TEST / "t2"), "-p", "kp"]),
    ("v3 module kpsc",   ["kleborate", "-a", str(one), "-o", str(TEST / "t3"),
                          "-m", "klebsiella_pneumo_complex"]),
    ("v2 --all",         ["kleborate", "-a", str(one), "--all", "-o", str(TEST / "t4.txt")]),
]

WORKING_CMD = None
for label, cmd in variants:
    print(f"--- trying: {label}\n    {' '.join(cmd)}")
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)
    if r.returncode == 0:
        produced = [p for p in TEST.rglob("*") if p.is_file() and p.stat().st_size > 0]
        print(f"    OK  -> {[p.name for p in produced][:6]}")
        WORKING_CMD = label
        break
    print(f"    rc={r.returncode}")
    print("    stderr:", (r.stderr or "").strip()[-800:] or "(none)")
    print("    stdout:", (r.stdout or "").strip()[-400:] or "(none)")

if WORKING_CMD is None:
    raise SystemExit("All variants failed - copy everything above and send it back.")

print(f"\n>>> using: {WORKING_CMD}")

# Show the output shape so we know the column names before the long run.
import pandas as pd
for p in sorted(TEST.rglob("*")):
    if p.is_file() and p.suffix in {".txt", ".tsv"} and p.stat().st_size > 0:
        df = pd.read_csv(p, sep="\t", dtype=str)
        print(f"\n{p.name}: {df.shape[0]} rows x {df.shape[1]} cols")
        print("columns:", list(df.columns))
        break

## Cell 8 — Full Kleborate run

Uses whichever command form passed the smoke test. Chunked and resumable; a failing chunk
is reported and skipped rather than killing the run.

In [ ]:
import subprocess, time
from pathlib import Path

OUT = WORK / "kleborate"
OUT.mkdir(exist_ok=True)
CHUNK = 20

def build_cmd(chunk, tag):
    files_ = list(map(str, chunk))
    if WORKING_CMD == "v3 preset kpsc":
        return ["kleborate", "-a", *files_, "-o", str(tag), "-p", "kpsc"]
    if WORKING_CMD == "v3 preset kp":
        return ["kleborate", "-a", *files_, "-o", str(tag), "-p", "kp"]
    if WORKING_CMD == "v3 module kpsc":
        return ["kleborate", "-a", *files_, "-o", str(tag),
                "-m", "klebsiella_pneumo_complex"]
    return ["kleborate", "-a", *files_, "--all", "-o", str(tag) + ".txt"]

fastas = sorted(ASM.glob("*.fna"))
chunks = [fastas[i:i + CHUNK] for i in range(0, len(fastas), CHUNK)]
print(f"{len(fastas)} genomes in {len(chunks)} chunks using '{WORKING_CMD}'\n")

failed = []
t0 = time.time()
for n, chunk in enumerate(chunks, 1):
    tag = OUT / f"chunk_{n:03d}"
    done = OUT / f"chunk_{n:03d}.done"
    if done.exists():
        print(f"chunk {n:>3}/{len(chunks)}  cached")
        continue

    r = subprocess.run(build_cmd(chunk, tag), capture_output=True, text=True)
    if r.returncode != 0:
        failed.append(n)
        print(f"chunk {n:>3}/{len(chunks)}  FAILED rc={r.returncode}")
        print("   ", (r.stderr or "").strip()[-600:])
        continue

    done.touch()
    el = time.time() - t0
    print(f"chunk {n:>3}/{len(chunks)}  ok  "
          f"[{el/60:.1f} min elapsed, ~{(el/n)*(len(chunks)-n)/60:.1f} min left]")

print(f"\nTotal: {(time.time()-t0)/60:.1f} min")
if failed:
    print(f"Failed chunks ({len(failed)}): {failed} - re-run this cell to retry them.")

## Cell 9 — Collect results into one table

In [ ]:
import pandas as pd

frames = []
for p in sorted(OUT.rglob("*")):
    if p.is_file() and p.suffix in {".txt", ".tsv"} and p.stat().st_size > 0:
        try:
            frames.append(pd.read_csv(p, sep="\t", dtype=str))
        except Exception as e:
            print(f"skip {p.name}: {e}")

if not frames:
    raise SystemExit("No Kleborate output found - check Cell 8 output.")

kleb = pd.concat(frames, ignore_index=True).drop_duplicates()
print(f"{len(kleb)} rows, {len(kleb.columns)} columns")
print("\ncolumns:", list(kleb.columns))
kleb.head(3)

## Cell 10 — The K-locus distribution

The headline result: which capsule types dominate Pakistani clinical *K. pneumoniae*.

In [ ]:
def pick(df, *candidates):
    """Column names shift between Kleborate versions; take the first that exists."""
    for c in candidates:
        if c in df.columns:
            return c
    lowered = {c.lower().replace(" ", "_"): c for c in df.columns}
    for c in candidates:
        k = c.lower().replace(" ", "_")
        if k in lowered:
            return lowered[k]
    return None

col_strain = pick(kleb, "strain", "Genome Name", "Name", "assembly")
col_st     = pick(kleb, "ST", "MLST ST", "st")
col_k      = pick(kleb, "K_locus", "K locus", "Best match locus", "K_type")
col_kconf  = pick(kleb, "K_locus_confidence", "K locus confidence", "Match confidence")
col_o      = pick(kleb, "O_locus", "O locus", "O_type")

print(f"strain={col_strain}  ST={col_st}  K={col_k}  Kconf={col_kconf}  O={col_o}\n")

if col_k:
    kdist = kleb[col_k].fillna("unknown").value_counts()
    print(f"=== K-locus distribution ({kdist.size} distinct types) ===")
    print(kdist.head(25).to_string())
    print(f"\nTop 10 K-loci cover {kdist.head(10).sum()/kdist.sum()*100:.1f}% of the cohort")

for label, col in [("K-locus call confidence", col_kconf),
                   ("O-locus distribution", col_o),
                   ("Sequence types", col_st)]:
    if col:
        print(f"\n=== {label} ===")
        print(kleb[col].fillna("unknown").value_counts().head(15).to_string())

if col_st and col_k:
    print("\n=== ST x K-locus (top pairings) ===")
    print(kleb.groupby([col_st, col_k]).size()
          .sort_values(ascending=False).head(20).to_string())

## Cell 11 — Save and download

Save into `D:\bacteriophage-data\processed\` on the laptop, then run
`scripts/07_join_typing.py`.

In [ ]:
from google.colab import files

# Normalise the join key to the bare accession (e.g. GCA_123456789.1).
if col_strain:
    kleb["assembly"] = (kleb[col_strain].astype(str)
                        .str.replace(r"\.(fna|fa|fasta)$", "", regex=True)
                        .str.strip())
    print("join key sample:", kleb["assembly"].head(3).tolist())

out_csv = "/content/kleborate_pakistan_kp.csv"
kleb.to_csv(out_csv, index=False)
print(f"wrote {len(kleb)} rows -> {out_csv}")

files.download(out_csv)